# ARC Prize 2026: Hierarchical Sub-Goal Decomposition Solver

## 1. Project Overview
This notebook implements a deterministic, search-based program synthesis solver for the ARC-AGI benchmark. It uses a **Hierarchical Sub-Goal Decomposition** approach:
- **State Representation**: Extracts topological, spatial, and geometric properties.
- **Sub-Goal Generation**: Computes state differences between input/output training pairs to dynamically propose intermediate property goals (e.g. `fill_enclosed_region`, `increase_symmetry`).
- **Hierarchical Planning**: Uses A* search to synthesize composable multi-step program pipelines up to depth $k=4$, guided by property heuristics.

This solver does not use LLMs, external ML models, or network calls, and strictly relies on deterministic rule execution.


## 2. Environment Setup
We append our dataset repository to `sys.path` so we can import our solver directly.

In [ ]:
import json
import sys
import time
from pathlib import Path

# Assuming the repo is mounted as a Kaggle Dataset at this path:
REPO_DATASET_PATH = "/kaggle/input/arc-reasoning-agent"
if REPO_DATASET_PATH not in sys.path:
    sys.path.insert(0, REPO_DATASET_PATH)

# Print sys.path to verify
print("sys.path:", sys.path[:3])

## 3. Load Competition Data

In [ ]:
from src.data.loader import load_task_from_dict

# Standard ARC Prize 2024 / ARC-AGI-2 test path
TEST_DATA_PATH = Path("/kaggle/input/arc-prize-2024/arc-agi_test_challenges.json")

# Fallback for local testing if running outside Kaggle
if not TEST_DATA_PATH.exists():
    print("Warning: Kaggle test path not found, falling back to local ARC-AGI-2 eval data.")
    TEST_DATA_PATH = Path(REPO_DATASET_PATH) / "data" / "ARC-AGI-2" / "data" / "evaluation"
    
tasks = {}
if TEST_DATA_PATH.is_dir():
    from src.data.loader import load_dataset
    tasks = load_dataset(TEST_DATA_PATH, recursive=True)
else:
    with open(TEST_DATA_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    for task_id, task_dict in raw_data.items():
        tasks[task_id] = load_task_from_dict(task_id, task_dict, is_test=True)
        
print(f"Loaded {len(tasks)} test tasks.")

## 4. Run Inference & Generate Predictions
We use the `generate_submission_dict` utility which processes all tasks, applies the `RuleBasedHierarchicalSolver_v1`, and automatically provides valid grid fallbacks if a task fails or times out.

In [ ]:
from src.submission.submission_generator import generate_submission_dict

print("Starting inference...")
submission, report = generate_submission_dict(tasks)
print("Inference complete.")

## 5. Validate and Save Submission

In [ ]:
from src.submission.validation import validate_submission

try:
    validate_submission(submission, expected_tasks=tasks)
    print("Validation PASSED.")
except Exception as e:
    print(f"Validation FAILED: {e}")

OUTPUT_PATH = "/kaggle/working/submission.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f)
print(f"Saved submission to {OUTPUT_PATH}")

## 6. Execution Summary

In [ ]:
print("="*50)
print("SUBMISSION RUN REPORT")
print("="*50)
print(f"Total Tasks:         {report['total_tasks']}")
print(f"Successful Search:   {report['successful_tasks']}")
print(f"Failed / Fallbacks:  {report['failed_tasks']}")
print(f"Total Runtime (s):   {report['runtime_seconds']}")
print("="*50)